# Greek LoRA fine-tuning on Colab: sensitivity run (Q3)
Same as the main run, but regularised against overfitting: **2 epochs, learning rate 2e-4, SpecAugment**. Settings are fixed in advance, not tuned on the test or rated sentences. Compare on dev loss and dev WER only.

Runtime → Change runtime type → **T4 GPU**. Then run the cells in order.
You upload only `colab_package.zip` (code, evaluation-set IDs); the Greek FLEURS train audio is downloaded here.
The result is `small-lora-el-reg.zip`: unzip it into `models/small-lora-el-reg/` in the project and transcribe locally.

In [ ]:
!nvidia-smi | head -15

## 1. Environment (same versions as the laptop)

In [ ]:
# Colab ships torchao 0.10, which this peft version rejects; we do not use it
!pip -q uninstall -y torchao
!pip -q install transformers==5.22.1 peft==0.21.2 ctranslate2==4.8.2 whisper-normalizer jiwer soundfile huggingface_hub tabulate pandas

## 2. Upload `colab_package.zip` and unpack it

In [ ]:
from google.colab import files
uploaded = files.upload()   # pick colab_package.zip
!mkdir -p /content/project && cd /content/project && unzip -oq /content/colab_package.zip && ls

## 3. Greek FLEURS: train and dev audio (about 2 GB)

In [ ]:
%cd /content/project
!python src/prepare_fleurs.py --langs el_gr --splits train dev

## 4. Smoke test (about 2 minutes). Check it trains and exports before the long run.

In [ ]:
!python src/finetune_lora.py --manifest data/manifests/fleurs_el_gr.csv --name smoke --max-train 64 --epochs 1 --spec-augment --dev-wer-items 0 2>&1 | grep -v -i warn | tail -8

## 5. The real run (2 epochs, learning rate 2e-4, SpecAugment)
Progress is printed every 10 steps. `models/small-lora-el-reg/training_log.csv` gets a row per epoch.

In [ ]:
!rm -rf models/smoke
!python -u src/finetune_lora.py --manifest data/manifests/fleurs_el_gr.csv --name small-lora-el-reg --epochs 2 --lr 2e-4 --spec-augment 2>&1 | grep -v -i warn

## 6. Download the result

In [ ]:
!cat models/small-lora-el-reg/training_log.csv
!cd models && zip -rq /content/small-lora-el-reg.zip small-lora-el-reg/ct2 small-lora-el-reg/adapter small-lora-el-reg/training_log.csv small-lora-el-reg/training_config.json
from google.colab import files
files.download("/content/small-lora-el-reg.zip")

If the download is slow or fails, copy it to Google Drive instead:
```python
from google.colab import drive; drive.mount('/content/drive')
!cp /content/small-lora-el-reg.zip /content/drive/MyDrive/
```